# Poglavje 7: Izdelava klepetalnih aplikacij
## Hitri začetek z OpenAI API-jem

Ta zvezek je prilagojen iz [Azure OpenAI vzorčnih primerov](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ki vključujejo zvezke za dostop do [Azure OpenAI](aoai-assignment.ipynb) storitev.

Python OpenAI API deluje tudi z Azure OpenAI modeli, z nekaj spremembami. Več o razlikah si lahko preberete tukaj: [Kako preklopiti med OpenAI in Azure OpenAI končnimi točkami s Pythonom](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Pregled  
"Veliki jezikovni modeli so funkcije, ki preslikajo besedilo v besedilo. Glede na vhodno niz besedila, velik jezikovni model poskuša napovedati besedilo, ki bo sledilo"(1). Ta "quickstart" zvezek bo uporabnike uvedel v osnovne pojme LLM, osnovne zahteve paketov za začetek z AML, rahlo uvod v oblikovanje pozivov ter več kratkih primerov različnih uporabnih primerov. 


## Kazalo vsebine  

[Pregled](#overview)  
[Kako uporabljati storitev OpenAI](#how-to-use-openai-service)  
[1. Ustvarjanje vaše OpenAI storitve](#1.-creating-your-openai-service)  
[2. Namestitev](#2.-installation)    
[3. Pooblastila](#3.-credentials)  

[Uporabniški primeri](#use-cases)    
[1. Povzetek besedila](#1.-summarize-text)  
[2. Razvrščanje besedila](#2.-classify-text)  
[3. Generiranje novih imen izdelkov](#3.-generate-new-product-names)  
[4. Fino nastavljanje razvrščevalca](#4.fine-tune-a-classifier)  

[Reference](#references)


### Ustvarite svoj prvi poziv  
Ta kratek primer bo zagotovil osnovni uvod v pošiljanje pozivov modelu OpenAI za preprosto nalogo »povzetek«.


**Koraki**:  
1. Namestite knjižnico OpenAI v vaše Python okolje  
2. Naložite standardne pomožne knjižnice in nastavite svoje običajne varnostne poverilnice OpenAI za storitev OpenAI, ki ste jo ustvarili  
3. Izberite model za svojo nalogo  
4. Ustvarite preprost poziv za model  
5. Pošljite svojo zahtevo modelu prek API-ja!


### 1. Namestite OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Uvozite pomožne knjižnice in ustvarite poverilnice


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Iskanje pravega modela  
Modeli, kot sta GPT-4o in GPT-4o mini, lahko razumejo in ustvarjajo naravni jezik ter so na voljo na platformi OpenAI z različnimi ravnmi zmogljivosti in hitrosti, primernimi za različne naloge.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Oblikovanje pozivov  

"Čar velikih jezikovnih modelov je v tem, da se s treningom za minimizacijo te napake napovedi čez ogromne količine besedila modeli na koncu naučijo pojmov, uporabnih za te napovedi. Na primer, naučijo se pojmov, kot so"(1):

* kako črkovati
* kako deluje slovnica
* kako parafrazirati
* kako odgovarjati na vprašanja
* kako voditi pogovor
* kako pisati v mnogih jezikih
* kako kodirati
* itd.

#### Kako nadzorovati velik jezikovni model  
"Od vseh vhodov v velik jezikovni model je daleč najbolj vpliven tekstovni poziv(1).

Velike jezikovne modele lahko spodbudimo k ustvarjanju izhoda na nekaj načinov:

Navodilo: Povej modelu, kaj želiš
Dokončanje: Spodbuja model, da dokonča začetek tega, kar želiš
Demonstracija: Pokaži modelu, kaj želiš, z:
nekaj primeri v pozivu
mnogimi stotimi ali tisoči primerov v natančno prilagojenem naboru podatkov za trening"



#### Obstajajo tri osnovna pravila za ustvarjanje pozivov:

**Pokaži in povej**. Jasno izrazi, kaj želiš, bodisi z navodili, primeri ali kombinacijo obojega. Če želiš, da model razvrsti seznam predmetov po abecedi ali razvrsti odstavek glede na čustva, mu pokaži, da je to tisto, kar želiš.

**Ponudi kakovostne podatke**. Če poskušaš ustvariti klasifikator ali dobiti model, da sledi vzorcu, se prepričaj, da je dovolj primerov. Prepiši svoje primere — model je običajno dovolj pameten, da prepozna osnovne črkovalne napake in ti odgovori, lahko pa tudi predpostavi, da je to namerno, kar lahko vpliva na odgovor.

**Preveri svoje nastavitve.** Nastavitve temperature in top_p nadzorujejo, kako determinističen je model pri generiranju odgovora. Če želiš odgovor, kjer je samo eden pravilen, jih nastavi nižje. Če želiš bolj raznolike odgovore, jih nastavi višje. Najpogostejša napaka pri teh nastavitvah je, da ljudje mislijo, da so to kontrole za "pametnost" ali "kreativnost".


Vir: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Pošlji!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ponovite isti klic, kako se rezultati primerjajo?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Povzemanje besedila  
#### Izziv  
Povzemite besedilo tako, da na konec besedilnega odstavka dodate 'tl;dr:'. Opazite, kako model razume, kako opraviti več nalog brez dodatnih navodil. Lahko eksperimentirate z bolj opisnimi namigi kot tl;dr, da prilagodite vedenje modela in prilagodite povzemanje, ki ga prejmete(3).  

Nedavno delo je pokazalo znatne izboljšave na mnogih nalogah in merilih NLP z vnaprejšnjim učenjem na veliki zbirki besedil, ki ji sledi fino nastavitev za specifično nalogo. Čeprav je arhitektura običajno nalogi nevtralna, ta metoda še vedno zahteva nalogno specifične podatkovne nize za fino nastavitev s tisoči ali deset tisoči primerov. Nasprotno pa ljudje na splošno lahko opravijo novo jezikovno nalogo že z nekaj primeri ali preprostimi navodili – nekaj, s čimer se trenutni NLP sistemi še vedno večinoma težko spopadajo. Tukaj pokažemo, da povečanje velikosti jezikovnih modelov zelo izboljša nalogno nevtralno, učenje z majhnim številom primerov, včasih pa doseže celo konkurenčnost s prejšnjimi najsodobnejšimi pristopi fino nastavitve.  



Povzeto  


# Vaje za več primerov uporabe  
1. Povzemanje besedila  
2. Razvrščanje besedila  
3. Generiranje novih imen izdelkov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Razvrsti besedilo  
#### Izziv  
Razvrsti predmete v kategorije, ki jih posredujemo ob inferenci. V naslednjem primeru v pozivu (*playground_reference) posredujemo tako kategorije kot besedilo za razvrščanje. 

Poizvedba stranke: Pozdravljeni, zadnje čase se je na tipkovnici mojega prenosnika zlomila ena tipka in potreboval bom nadomestno:

Razvrščena kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Ustvari nova imena izdelkov
#### Izziv
Ustvarite imena izdelkov na podlagi primerov besed. Tukaj v poziv vključimo informacije o izdelku, za katerega bomo ustvarjali imena. Prav tako podamo podoben primer, da pokažemo vzorec, ki ga želimo dobiti. Nastavili smo tudi visoko vrednost temperature, da povečamo naključnost in bolj inovativne odzive.

Opis izdelka: Domač izdelovalec milkshake-ov
Izvorne besede: hiter, zdrav, kompakten.
Imena izdelkov: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis izdelka: Par čevljev, ki se prilegata vsaki velikosti stopala.
Izvorne besede: prilagodljiv, prilegajoč, omni-prilegajoč.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najboljše prakse za fino uravnavanje GPT modelov za klasifikacijo besedil](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za več pomoči  
[Ekipa za komercializacijo OpenAI](AzureOpenAITeam@microsoft.com) 


# Sodelavci
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
